Enrollment No.: 240840116041

## **Practical 9 |Design and develop a simple interactive dashboard using Dash/Plotly for business reporting.**

**Step 1: Install libraries**

In [1]:
# Install the required libraries
!pip install -q dash plotly pandas

**Step 2: Import libraries**

In [2]:
# Import Pandas to handle data
import pandas as pd

# Import Plotly Express to create graphs
import plotly.express as px

# Import Dash components to create the dashboard
from dash import Dash, dcc, html, Input, Output

# Import files to upload CSV
from google.colab import files

**Step 3: Load the Pizza Sales dataset**

In [3]:
# Upload the pizza sales dataset
uploaded = files.upload()

# Read the CSV file
df = pd.read_csv("pizza_sales.csv")

# Convert order_date into date format
df["order_date"] = pd.to_datetime(
    df["order_date"], dayfirst=True
)

# Create month columns
df["month"] = df["order_date"].dt.month_name()
df["month_number"] = df["order_date"].dt.month

# Display the dataset
print(df.head())

# Display dataset information
print("\nDataset Shape:", df.shape)

# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

Saving pizza_sales.csv to pizza_sales.csv
   pizza_id  order_id  pizza_name_id  quantity order_date order_time  \
0         1         1     hawaiian_m         1 2015-01-01   11:38:36   
1         2         2  classic_dlx_m         1 2015-01-01   11:57:40   
2         3         2  five_cheese_l         1 2015-01-01   11:57:40   
3         4         2    ital_supr_l         1 2015-01-01   11:57:40   
4         5         2     mexicana_m         1 2015-01-01   11:57:40   

   unit_price  total_price pizza_size pizza_category  \
0       13.25        13.25          M        Classic   
1       16.00        16.00          M        Classic   
2       18.50        18.50          L         Veggie   
3       20.75        20.75          L        Supreme   
4       16.00        16.00          M         Veggie   

                                   pizza_ingredients  \
0           Sliced Ham, Pineapple, Mozzarella Cheese   
1  Pepperoni, Mushrooms, Red Onions, Red Peppers,...   
2  Mozzarella Cheese

**Step 4: Calculate KPI values**

In [4]:
# Calculate KPI values

total_revenue = df["total_price"].sum()

average_order_id = df["order_id"].mean()

total_quantity = df["quantity"].sum()

average_price = df["total_price"].mean()

print("Total Revenue:", total_revenue)
print("Average Order ID:", average_order_id)
print("Total Quantity:", total_quantity)
print("Average Price:", average_price)

print("All KPI values calculated successfully!")

Total Revenue: 817860.05
Average Order ID: 10701.479761415056
Total Quantity: 49574
Average Price: 16.821473673385437
All KPI values calculated successfully!


**Step 4: Create the dashboard layout**

In [5]:
# Calculate KPI values
total_revenue = df["total_price"].sum()
average_order_id = df["order_id"].mean()
total_quantity = df["quantity"].sum()
average_price = df["total_price"].mean()


# Prepare Monthly Revenue Data
monthly = (
    df.groupby(df["order_date"].dt.month)["total_price"]
    .sum()
    .reset_index()
)
monthly.columns = ["month_number", "total_price"]
monthly["month"] = pd.to_datetime(
    monthly["month_number"], format="%m"
).dt.strftime("%B")
monthly = monthly.sort_values("month_number")


# Prepare Category Data
category = df.groupby("pizza_category")["total_price"].sum().reset_index()


# Prepare Pizza Size Data
size = df.groupby("pizza_size")["total_price"].sum().reset_index()
size_order = ["S", "M", "L", "XL", "XXL"]
size["pizza_size"] = pd.Categorical(
    size["pizza_size"], categories=size_order, ordered=True
)
size = size.sort_values("pizza_size")


# Create Monthly Revenue Line Chart
fig1 = px.line(monthly, x="month", y="total_price")
fig1.update_traces(line=dict(color="#1689f8", width=2), line_shape="spline")
fig1.update_layout(
    title="Sum of total_price by Month",
    xaxis_title="Month",
    yaxis_title="",
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False
)


# Create Pizza Category Donut Chart
fig2 = px.pie(
    category,
    names="pizza_category",
    values="total_price",
    hole=0.55,
    color_discrete_sequence=["#2026a5", "#f06432", "#1689f8", "#79008f"]
)
fig2.update_traces(
    textposition="outside",
    texttemplate="%{value:.2s} (%{percent})"
)
fig2.update_layout(
    title="Sales by Pizza Category",
    annotations=[dict(
        text=f"{total_revenue/1000:.2f}K",
        x=0.5, y=0.5, showarrow=False, font_size=16
    )],
    plot_bgcolor="white",
    paper_bgcolor="white"
)


# Create Pizza Size Horizontal Bar Chart
fig3 = px.bar(
    size,
    x="total_price",
    y="pizza_size",
    orientation="h",
    color_discrete_sequence=["#1689f8"]
)
fig3.update_layout(
    title="Sales by Pizza Size",
    xaxis_title="",
    yaxis_title="",
    plot_bgcolor="white",
    paper_bgcolor="white"
)


# Create Dash Application
app = Dash(__name__)


# KPI Card Design (original styling retained)
def kpi_card(title, value):
    return html.Div(
        [
            html.P(
                title,
                style={
                    "fontSize": "10px",
                    "color": "#555",
                    "margin": "0 0 3px 0"
                }
            ),
            html.H3(
                value,
                style={
                    "fontSize": "16px",
                    "color": "#222",
                    "margin": "0"
                }
            )
        ],
        style={
            "backgroundColor": "white",
            "border": "1px solid #aaa",
            "borderRadius": "5px",
            "padding": "12px 16px",
            "minHeight": "64px",
            "height": "100%",
            "width": "100%",
            "boxSizing": "border-box",
            "flex": "1"
        }
    )


# Reusable filter dropdown styling
filter_style = {
    "backgroundColor": "white",
    "border": "1px solid #aaa",
    "borderRadius": "5px",
    "padding": "8px",
    "flex": "1",
    "minWidth": "180px"
}


# Dashboard Layout
app.layout = html.Div(
    [
        # Dashboard Header (original design retained)
        html.Div(
            "🍕 Pizza Sales Dashboard",
            style={
                "backgroundColor": "#991b1b",
                "color": "white",
                "textAlign": "center",
                "fontSize": "30px",
                "padding": "8px",
                "fontFamily": "Georgia"
            }
        ),

        # New interactive filters
        html.Div(
            [
                html.Div(
                    [
                        html.Label("Month", style={"fontWeight": "bold", "fontSize": "12px"}),
                        dcc.Dropdown(
                            id="month-filter",
                            options=[{"label": "All Months", "value": "All"}] +
                                    [{"label": m, "value": m} for m in
                                     ["January", "February", "March", "April", "May", "June",
                                      "July", "August", "September", "October", "November", "December"]],
                            value="All",
                            clearable=False
                        )
                    ],
                    style=filter_style
                ),
                html.Div(
                    [
                        html.Label("Pizza Category", style={"fontWeight": "bold", "fontSize": "12px"}),
                        dcc.Dropdown(
                            id="category-filter",
                            options=[{"label": "All Categories", "value": "All"}] +
                                    [{"label": str(v), "value": str(v)} for v in
                                     sorted(df["pizza_category"].dropna().unique())],
                            value="All",
                            clearable=False
                        )
                    ],
                    style=filter_style
                ),
                html.Div(
                    [
                        html.Label("Pizza Size", style={"fontWeight": "bold", "fontSize": "12px"}),
                        dcc.Dropdown(
                            id="size-filter",
                            options=[{"label": "All Sizes", "value": "All"}] +
                                    [{"label": str(v), "value": str(v)} for v in
                                     ["S", "M", "L", "XL", "XXL"] if v in df["pizza_size"].astype(str).unique()],
                            value="All",
                            clearable=False
                        )
                    ],
                    style=filter_style
                ),
                html.Div(
                    [
                        html.Label("Pizza Name", style={"fontWeight": "bold", "fontSize": "12px"}),
                        dcc.Dropdown(
                            id="pizza-filter",
                            options=[{"label": "All Pizzas", "value": "All"}] +
                                    [{"label": str(v), "value": str(v)} for v in
                                     sorted(df["pizza_name"].dropna().unique())],
                            value="All",
                            clearable=False,
                            searchable=True
                        )
                    ],
                    style=filter_style
                )
            ],
            style={
                "display": "flex",
                "flexWrap": "wrap",
                "gap": "10px",
                "margin": "10px",
                "padding": "10px",
                "backgroundColor": "#eeeeee"
            }
        ),

        # First KPI Row (same styling; values update with filters)
        html.Div(
            [
                html.Div(
                    kpi_card("Sum of total_price", f"{total_revenue/1000:.2f}K"),
                    id="kpi-revenue",
                    style={"flex": "1", "minWidth": "0"}
                ),
                html.Div(
                    kpi_card("Average of order_id", f"{average_order_id/1000:.2f}K"),
                    id="kpi-order-id",
                    style={"flex": "1", "minWidth": "0"}
                )
            ],
            style={
                "display": "flex",
                "gap": "10px",
                "margin": "10px",
                "width": "calc(100% - 20px)",
                "alignItems": "stretch"
            }
        ),

        # Second KPI Row
        html.Div(
            [
                html.Div(
                    kpi_card("Sum of quantity", f"{total_quantity/1000:.0f}K"),
                    id="kpi-quantity",
                    style={"flex": "1", "minWidth": "0"}
                ),
                html.Div(
                    kpi_card("Average of total_price", f"{average_price:.2f}"),
                    id="kpi-average-price",
                    style={"flex": "1", "minWidth": "0"}
                )
            ],
            style={
                "display": "flex",
                "gap": "10px",
                "margin": "10px",
                "width": "calc(100% - 20px)",
                "alignItems": "stretch"
            }
        ),

        # Monthly Revenue Chart
        html.Div(
            dcc.Graph(
                id="monthly-chart",
                figure=fig1,
                config={"displayModeBar": True, "displaylogo": False}
            ),
            style={
                "backgroundColor": "white",
                "border": "1px solid #aaa",
                "borderRadius": "5px",
                "margin": "10px"
            }
        ),

        # Category and Size Charts
        html.Div(
            [
                html.Div(
                    dcc.Graph(
                        id="category-chart",
                        figure=fig2,
                        config={"displayModeBar": True, "displaylogo": False}
                    ),
                    style={
                        "flex": "1",
                        "minWidth": "0",
                        "backgroundColor": "white",
                        "border": "1px solid #aaa",
                        "borderRadius": "5px"
                    }
                ),
                html.Div(
                    dcc.Graph(
                        id="size-chart",
                        figure=fig3,
                        config={"displayModeBar": True, "displaylogo": False}
                    ),
                    style={
                        "flex": "1",
                        "minWidth": "0",
                        "backgroundColor": "white",
                        "border": "1px solid #aaa",
                        "borderRadius": "5px"
                    }
                )
            ],
            style={"display": "flex", "gap": "10px", "margin": "10px"}
        )
    ],
    style={
        "backgroundColor": "#eeeeee",
        "padding": "3px",
        "fontFamily": "Arial",
        "minHeight": "100vh"
    }
)

print("Dashboard layout created successfully!")


Dashboard layout created successfully!


**Step 5: Add interactive dashboard functionality**

In [6]:
# Add interactive filtering: all KPIs and charts update together
@app.callback(
    [
        Output("kpi-revenue", "children"),
        Output("kpi-order-id", "children"),
        Output("kpi-quantity", "children"),
        Output("kpi-average-price", "children"),
        Output("monthly-chart", "figure"),
        Output("category-chart", "figure"),
        Output("size-chart", "figure")
    ],
    [
        Input("month-filter", "value"),
        Input("category-filter", "value"),
        Input("size-filter", "value"),
        Input("pizza-filter", "value")
    ]
)
def update_dashboard(selected_month, selected_category, selected_size, selected_pizza):
    filtered_df = df.copy()

    if selected_month != "All":
        filtered_df = filtered_df[filtered_df["month"] == selected_month]
    if selected_category != "All":
        filtered_df = filtered_df[filtered_df["pizza_category"] == selected_category]
    if selected_size != "All":
        filtered_df = filtered_df[filtered_df["pizza_size"].astype(str) == selected_size]
    if selected_pizza != "All":
        filtered_df = filtered_df[filtered_df["pizza_name"] == selected_pizza]

    # Filtered KPI values
    revenue = filtered_df["total_price"].sum()
    avg_order_id = filtered_df["order_id"].mean()
    quantity = filtered_df["quantity"].sum()
    avg_price = filtered_df["total_price"].mean()

    # Avoid displaying NaN when the selected combination has no records
    avg_order_id_text = f"{avg_order_id/1000:.2f}K" if pd.notna(avg_order_id) else "N/A"
    avg_price_text = f"{avg_price:.2f}" if pd.notna(avg_price) else "N/A"

    revenue_card = kpi_card("Sum of total_price", f"{revenue/1000:.2f}K")
    order_id_card = kpi_card("Average of order_id", avg_order_id_text)
    quantity_card = kpi_card("Sum of quantity", f"{quantity/1000:.0f}K")
    average_price_card = kpi_card("Average of total_price", avg_price_text)

    # Monthly revenue chart
    monthly_filtered = (
        filtered_df.groupby("month_number")["total_price"]
        .sum()
        .reset_index()
        .sort_values("month_number")
    )
    monthly_filtered["month"] = pd.to_datetime(
        monthly_filtered["month_number"], format="%m"
    ).dt.strftime("%B")

    monthly_fig = px.line(monthly_filtered, x="month", y="total_price")
    monthly_fig.update_traces(
        line=dict(color="#1689f8", width=2),
        line_shape="spline",
        hovertemplate="Month: %{x}<br>Revenue: %{y:.2f}<extra></extra>"
    )
    monthly_fig.update_layout(
        title="Sum of total_price by Month",
        xaxis_title="Month",
        yaxis_title="",
        plot_bgcolor="white",
        paper_bgcolor="white",
        showlegend=False,
        hovermode="x unified",
        dragmode="zoom"
    )

    # Category donut chart
    category_filtered = (
        filtered_df.groupby("pizza_category")["total_price"]
        .sum()
        .reset_index()
    )
    category_fig = px.pie(
        category_filtered,
        names="pizza_category",
        values="total_price",
        hole=0.55,
        color_discrete_sequence=["#2026a5", "#f06432", "#1689f8", "#79008f"]
    )
    category_fig.update_traces(
        textposition="outside",
        texttemplate="%{value:.2s} (%{percent})",
        hovertemplate="Category: %{label}<br>Revenue: %{value:.2f}<br>Share: %{percent}<extra></extra>"
    )
    category_fig.update_layout(
        title="Sales by Pizza Category",
        annotations=[dict(
            text=f"{revenue/1000:.2f}K",
            x=0.5, y=0.5, showarrow=False, font_size=16
        )],
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    # Pizza size horizontal bar chart
    size_filtered = filtered_df.groupby("pizza_size")["total_price"].sum().reset_index()
    size_filtered["pizza_size"] = pd.Categorical(
        size_filtered["pizza_size"],
        categories=["S", "M", "L", "XL", "XXL"],
        ordered=True
    )
    size_filtered = size_filtered.sort_values("pizza_size")
    size_fig = px.bar(
        size_filtered,
        x="total_price",
        y="pizza_size",
        orientation="h",
        color_discrete_sequence=["#1689f8"]
    )
    size_fig.update_traces(
        hovertemplate="Size: %{y}<br>Revenue: %{x:.2f}<extra></extra>"
    )
    size_fig.update_layout(
        title="Sales by Pizza Size",
        xaxis_title="",
        yaxis_title="",
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    return (
        revenue_card,
        order_id_card,
        quantity_card,
        average_price_card,
        monthly_fig,
        category_fig,
        size_fig
    )

print("Interactive filters added successfully!")


Interactive filters added successfully!


**Step 6: Run the dashboard in Google Colab**

In [7]:
# Run the dashboard and display it inside Google Colab
app.run(
    jupyter_mode="inline",
    debug=False,
    port=8050
)

<IPython.core.display.Javascript object>